# Dual-Domain (Time + Freq) Fine-tuning

Based on the dual-domain SupCon pretrained model (time + frequency).
We finetune a classifier that uses both time and frequency features.

N defines the number of labeled samples per class for fine-tuning.

In [1]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
from __future__ import unicode_literals

import warnings
warnings.filterwarnings('ignore')
import numpy as np

from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
import torch
from torch import nn
import torch.nn.functional as F
from torch import optim
from torch.autograd import Variable
import tqdm
import pickle
import argparse
from torch.cuda.amp import GradScaler, autocast

import random
import sys
import os
import collections
from sklearn.model_selection import train_test_split

## GPU Allocation

In [2]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu", 0)
kwargs = {'num_workers': 0, 'pin_memory': True} if use_cuda else {}
print(f'Device: {device}')

Device: cuda:0


## Parameters

In [3]:
batch_size = 16

## Loading the Fine-tuning Datasets

Load both time domain and frequency domain data. Must be aligned.

In [4]:
DATASET = 'AWF'

if DATASET == 'AWF':
    # --- 时域数据 ---
    data_time = np.load('./datasets/awf2.npz')
    print('Time files:', data_time.files)
    x_time_total = data_time['data']
    y_total = data_time['labels']

    # --- 频域数据 ---
    data_freq = np.load('./datasets/awf2_freq.npz')
    print('Freq files:', data_freq.files)
    x_freq_total = data_freq['x']

print(f"Time shape: {x_time_total.shape}")
print(f"Freq shape: {x_freq_total.shape}")
print(f"Labels: {y_total.shape}")
assert len(x_time_total) == len(x_freq_total) == len(y_total), "Mismatched sample counts!"

num_classes = len(np.unique(y_total))
print(f"Number of classes: {num_classes}")

Time files: ['data', 'labels']
Freq files: ['x', 'y']
Time shape: (257500, 5000)
Freq shape: (257500, 2500)
Labels: (257500,)
Number of classes: 103


In [5]:
# 训练/测试拆分（保持时域和频域的对齐）
x_time_train, x_time_test, x_freq_train, x_freq_test, y_train, y_test = train_test_split(
    x_time_total, x_freq_total, y_total, test_size=0.2, random_state=42, stratify=y_total)

print(f"Train: time {x_time_train.shape}, freq {x_freq_train.shape}, y {y_train.shape}")
print(f"Test:  time {x_time_test.shape}, freq {x_freq_test.shape}, y {y_test.shape}")

Train: time (206000, 5000), freq (206000, 2500), y (206000,)
Test:  time (51500, 5000), freq (51500, 2500), y (51500,)


In [6]:
# 每个类别随机采样 N 个样本用于微调
def sample_traces(x_time, x_freq, y, N):
    train_index = []
    for c in range(num_classes):
        idx = np.where(y == c)[0]
        idx = np.random.choice(idx, min(N, len(idx)), False)
        train_index.extend(idx)
    train_index = np.array(train_index)
    np.random.shuffle(train_index)

    return x_time[train_index], x_freq[train_index], y[train_index]

## Backbone Model

Same DFNet as used in pretrain.

In [7]:
class DFNet(nn.Module):
    def __init__(self, out_dim, input_feature_dim):
        super(DFNet, self).__init__()
        kernel_size = 8
        conv_stride = 1
        pool_stride = 4
        pool_size = 8

        self.conv1 = nn.Conv1d(1, 32, kernel_size, stride=conv_stride)
        self.conv1_1 = nn.Conv1d(32, 32, kernel_size, stride=conv_stride)
        self.conv2 = nn.Conv1d(32, 64, kernel_size, stride=conv_stride)
        self.conv2_2 = nn.Conv1d(64, 64, kernel_size, stride=conv_stride)
        self.conv3 = nn.Conv1d(64, 128, kernel_size, stride=conv_stride)
        self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride=conv_stride)
        self.conv4 = nn.Conv1d(128, 256, kernel_size, stride=conv_stride)
        self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride=conv_stride)

        self.batch_norm1 = nn.BatchNorm1d(32)
        self.batch_norm2 = nn.BatchNorm1d(64)
        self.batch_norm3 = nn.BatchNorm1d(128)
        self.batch_norm4 = nn.BatchNorm1d(256)

        self.max_pool_1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_2 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_3 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_4 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)

        self.dropout1 = nn.Dropout(p=0.1)
        self.dropout2 = nn.Dropout(p=0.1)
        self.dropout3 = nn.Dropout(p=0.1)
        self.dropout4 = nn.Dropout(p=0.1)

        # 动态计算展平后的维度
        with torch.no_grad():
            dummy_input = torch.zeros(1, 1, input_feature_dim)
            out = self._forward_features(dummy_input)
            flattened_dim = out.view(1, -1).size(1)

        self.fc = nn.Linear(flattened_dim, out_dim)
        self.weight_init()

    def _forward_features(self, x):
        x = F.pad(x, (3,4))
        x = F.elu((self.conv1(x)))
        x = F.pad(x, (3,4))
        x = F.elu(self.batch_norm1(self.conv1_1(x)))
        x = F.pad(x, (3, 4))
        x = self.max_pool_1(x)
        x = self.dropout1(x)

        x = F.pad(x, (3,4))
        x = F.relu((self.conv2(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm2(self.conv2_2(x)))
        x = F.pad(x, (3,4))
        x = self.max_pool_2(x)
        x = self.dropout2(x)

        x = F.pad(x, (3,4))
        x = F.relu((self.conv3(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm3(self.conv3_3(x)))
        x = F.pad(x, (3,4))
        x = self.max_pool_3(x)
        x = self.dropout3(x)

        x = F.pad(x, (3,4))
        x = F.relu((self.conv4(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm4(self.conv4_4(x)))
        x = F.pad(x, (3,4))
        x = self.max_pool_4(x)
        x = self.dropout4(x)
        return x

    def weight_init(self):
        for n, m in self.named_modules():
            if isinstance(m, nn.Linear) or isinstance(m, nn.Conv1d):
                torch.nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    m.bias.data.zero_()

    def forward(self, inp):
        x = self._forward_features(inp)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## Dual-Domain Classifier (for finetuning)

Uses both time_encoder and freq_encoder from the pretrained SupCon model.
Concatenates their 512-dim outputs → 1024-dim → classifier head.

In [8]:
class DualDomainClassifier(nn.Module):
    def __init__(self, time_encoder, freq_encoder, num_classes):
        super().__init__()
        self.time_encoder = time_encoder
        self.freq_encoder = freq_encoder
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, x_time, x_freq):
        h_time = self.time_encoder(x_time)
        h_freq = self.freq_encoder(x_freq)
        out = torch.cat([h_time, h_freq], dim=1)
        return self.classifier(out)

## Data Loader

Returns aligned (time, freq, label) tuples.

In [9]:
class DualData(Dataset):
    def __init__(self, x_time, x_freq, y):
        self.x_time = x_time
        self.x_freq = x_freq
        self.y = y

    def __getitem__(self, index):
        return self.x_time[index], self.x_freq[index], self.y[index]

    def __len__(self):
        return len(self.y)

## Loading the Pre-trained Dual-Domain Model

We load the SupCon pretrained `time_encoder` and `freq_encoder` weights,
skip the projectors, and add a new classifier head.

**注意**: pretrain 用的 awf1（100类），finetune 用的 awf2（103类），
但 encoder 部分维度一致，可以直接加载。classifier 随机初始化。

In [10]:
def load_checkpoint():
    """
    Load dual-domain SupCon pretrained model.
    Extract time_encoder and freq_encoder weights, skip projectors.
    """
    time_input_dim = x_time_train.shape[1]  # 5000
    freq_input_dim = x_freq_train.shape[1]   # 2500

    # 创建两个 encoder
    time_encoder = DFNet(out_dim=512, input_feature_dim=time_input_dim)
    freq_encoder = DFNet(out_dim=512, input_feature_dim=freq_input_dim)

    # ===== 加载 dual-domain checkpoint =====
    checkpoint = torch.load('./checkpoints/supcon/WFTFC_dualsupcon_epoch_100.pth.tar')

    # 提取 time_encoder 权重
    time_state = {}
    for k, v in checkpoint.items():
        if k.startswith('time_encoder.'):
            new_k = k[len('time_encoder.'):]
            time_state[new_k] = v

    # 提取 freq_encoder 权重
    freq_state = {}
    for k, v in checkpoint.items():
        if k.startswith('freq_encoder.'):
            new_k = k[len('freq_encoder.'):]
            freq_state[new_k] = v

    # 加载 encoder 权重
    log_t = time_encoder.load_state_dict(time_state, strict=False)
    log_f = freq_encoder.load_state_dict(freq_state, strict=False)
    print(f"Time encoder missing: {log_t.missing_keys}")
    print(f"Freq encoder missing: {log_f.missing_keys}")

    # 组装分类器
    model = DualDomainClassifier(time_encoder, freq_encoder, num_classes).to(device)
    return model

## Init Test Data Loaders

In [11]:
test_dataset = DualData(x_time_test, x_freq_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, drop_last=True)

## Train and Test Functions

Both take `(x_time, x_freq, target)` instead of just `(data, target)`.

In [12]:
def train(model, device, train_loader, optimizer):
    model.train()
    for batch_idx, (x_time, x_freq, target) in enumerate(train_loader):
        x_time = x_time.view(x_time.size(0), 1, x_time.size(1)).float().to(device)
        x_freq = x_freq.view(x_freq.size(0), 1, x_freq.size(1)).float().to(device)
        target = target.to(device).long()

        optimizer.zero_grad()
        output = model(x_time, x_freq)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f"Loss: {loss.item():.6f}")

def test(model, device, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x_time, x_freq, target in loader:
            x_time = x_time.view(x_time.size(0), 1, x_time.size(1)).float().to(device)
            x_freq = x_freq.view(x_freq.size(0), 1, x_freq.size(1)).float().to(device)
            target = target.to(device).long()

            output = model(x_time, x_freq)
            output = torch.softmax(output, dim=1)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).float().sum().item()
    return correct / len(loader.dataset)

## Running for 5 Times

In [13]:
# N defines the number of labeled samples per class for fine-tuning
N = 5

In [14]:
accuracies = []
for _ in range(5):
    x_time_ft, x_freq_ft, y_ft = sample_traces(x_time_train, x_freq_train, y_train, N)
    print(f"Input size: time {x_time_ft.shape}, freq {x_freq_ft.shape}, y {y_ft.shape}")

    train_dataset = DualData(x_time_ft, x_freq_ft, y_ft)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    model = load_checkpoint()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    best_acc = 0
    for epoch in range(101):
        print(f"Epoch: {epoch}")
        train(model, device, train_loader, optimizer)

        acc = test(model, device, test_loader)
        best_acc = max(best_acc, acc)

        if epoch % 10 == 0:
            print(f"Accuracy on test dataset: {acc*100:.2f}")

    accuracies.append(best_acc)
    print('------------------------------------------------')

Input size: time (515, 5000), freq (515, 2500), y (515,)
Time encoder missing: []
Freq encoder missing: []
Epoch: 0
Loss: 5.552017
Accuracy on test dataset: 5.32
Epoch: 1
Loss: 4.135025
Epoch: 2
Loss: 3.669783
Epoch: 3
Loss: 2.924567
Epoch: 4
Loss: 1.259995
Epoch: 5
Loss: 0.433949
Epoch: 6
Loss: 0.209926
Epoch: 7
Loss: 0.110277
Epoch: 8
Loss: 0.108565
Epoch: 9
Loss: 0.007672
Epoch: 10
Loss: 0.012222
Accuracy on test dataset: 84.98
Epoch: 11
Loss: 0.014459
Epoch: 12
Loss: 0.042343
Epoch: 13
Loss: 0.025091
Epoch: 14
Loss: 0.026184
Epoch: 15
Loss: 0.002382
Epoch: 16
Loss: 0.005304
Epoch: 17
Loss: 0.004913
Epoch: 18
Loss: 0.004990
Epoch: 19
Loss: 0.006189
Epoch: 20
Loss: 0.004363
Accuracy on test dataset: 86.63
Epoch: 21
Loss: 0.001298
Epoch: 22
Loss: 0.001173
Epoch: 23
Loss: 0.018073
Epoch: 24
Loss: 0.002811
Epoch: 25
Loss: 0.005137
Epoch: 26
Loss: 0.000582
Epoch: 27
Loss: 0.002452
Epoch: 28
Loss: 0.005688
Epoch: 29
Loss: 0.004443
Epoch: 30
Loss: 0.000390
Accuracy on test dataset: 88.19
E

In [15]:
accuracies = np.array(accuracies)
print(f"Test accuracy on dual-domain: avg -> {np.mean(accuracies)*100:.1f}, std -> {np.std(accuracies)*100:.1f}")

Test accuracy on dual-domain: avg -> 89.7, std -> 0.5
